# zagg end to end: ICESat-2 photons to analysis-ready Zarr

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/sponsor_end_to_end.ipynb)

This notebook walks the **complete zagg pipeline** on real public data — NASA's
ICESat-2 ATL03 photon cloud over the NEON [SERC](https://www.neonscience.org/field-sites/serc)
site (Smithsonian Environmental Research Center, Edgewater MD):

1. **Catalog** (§1) — query NASA CMR for granule metadata and snapshot it as a
   STAC-geoparquet file (*build once, reuse everywhere*).
2. **Shardmap** (§2) — intersect granule footprints with the output grid, once,
   into a work-distribution manifest that carries a strict-AOI cell mask
   (issue #101).
3. **Two aggregations from that one shardmap** (§3–5, issue #89): a per-cell
   **t-digest** sketch of photon heights with a morton location channel
   (issue #87), then a per-chunk **gain/bias 128-bin waveform** — both through
   the virtual chunk-index seam (issue #160): the first run *persists* a
   granule-keyed chunk-manifest cache, and the second run *consumes* it via the
   `sidecar` backend.
4. **Read-back** (§6) — open the Zarr outputs, map the photon density, and
   check the t-digest reconstruction against the binned waveform.

The **companion notebook** [`sponsor_read_only.ipynb`](sponsor_read_only.ipynb)
consumes the published outputs **anonymously** (no credentials at all) and is the
one to hand to anyone who just wants to *read* the products.

**Requirements.** `pip install "zagg[analysis,catalog,viz]>=0.14"` (the Binder
image already provides zagg via the repo's `.binder/` environment), plus
`h5coro-hidefix` ≥ 0.2 for the `sidecar` index backend used in §5.
Sections 1–3 run **anywhere, with no credentials** — CMR granule *metadata* is
anonymous. The aggregation runs (§4–5) read ATL03 granule *data* and write to
S3, so they need NASA Earthdata Login + AWS credentials (each run cell states
exactly what it needs — §5 as executed here needs neither, reading a local
granule cache and writing locally).

> **Provisional versions note (July 2026).** The committed execution below ran
> against zagg `0.13.0` — the same #163 + #165 feature set that ships as
> **0.14.0**, which the pins above reference — and an `h5coro-hidefix` `0.2.0`
> wheel built from its `main` ahead of the PyPI release. If either pin fails
> to resolve yet, install zagg from `main` and h5coro-hidefix per its README.

## 1. Catalog: CMR → STAC-geoparquet, built once

`CMRSource` speaks directly to NASA's CMR-STAC endpoint — no Earthdata Login is
needed for granule-metadata queries. The result wraps a
[stac-geoparquet](https://github.com/stac-utils/stac-geoparquet) Arrow table
(one row per granule, S3 **and** HTTPS hrefs preserved).

The area of interest is the NEON SERC AOP flight box
(`tests/data/benchmark/AOP_NEON.geojson`) and the temporal window is the full
multi-year benchmark slice `2018-10-13 .. 2025-06-01` — the same AOI + window
the repo's Lambda benchmark matrix pins (`tests/data/benchmark/README.md`).

**Build once, commit the snapshot.** A CMR fetch is cheap here (~60 granules)
but O(20 min) for mission-scale AOIs, so the repo convention (issues #148/#152)
is to snapshot the catalog as geoparquet next to the artifacts that derive from
it, and rebuild only to deliberately re-pin. This notebook writes
`data/cat_serc_atl03_007.parquet`, which is committed with the repo.

In [ ]:
import zagg

print(f"zagg {zagg.__version__}")

In [ ]:
from zagg.catalog.sources import CMRSource, Query

AOI_GEOJSON = "../tests/data/benchmark/AOP_NEON.geojson"  # NEON SERC AOP box
START_DATE, END_DATE = "2018-10-13", "2025-06-01"         # benchmark temporal pin

query = Query(
    short_name="ATL03",
    version="007",
    start_date=START_DATE,
    end_date=END_DATE,
    region=AOI_GEOJSON,          # a bbox tuple also works
    provider="NSIDC_CPRD",
)

catalog = CMRSource().fetch(query)
print(f"Fetched {len(catalog)} ATL03 granules for {query.collection} over SERC")

In [ ]:
from pathlib import Path

from zagg.catalog.sources import Catalog

DATA = Path("data")
DATA.mkdir(exist_ok=True)
cat_path = DATA / "cat_serc_atl03_007.parquet"
catalog.to_geoparquet(str(cat_path))

# Round-trip: later runs load the committed snapshot instead of re-fetching
# CMR (the shipped-config guard is tests/test_config.py::TestSercExampleConfigs).
catalog = Catalog.from_geoparquet(str(cat_path))
print(f"{cat_path} ({cat_path.stat().st_size / 1024:.0f} KB), "
      f"{len(catalog)} granules on reload")

## 2. Shardmap: one manifest, built from the saved catalog

A **ShardMap** maps each populated grid *shard* (the dispatch unit — one worker
per shard) to the granules whose footprints cross it. It is tied to the grid's
**spatial signature only**, so any config that shares the spatial grid can reuse
it — which is exactly what section 5 does with two different aggregations
(issue #89).

The grid comes from the run configs themselves (single source of truth): a
HEALPix nested grid, `parent_order=10` shards (~5 km), `child_order=19` leaf
cells (~10 m), 64 inner Zarr chunks per shard. Both example configs turn on
`output.aoi_mask` (issue #101), so the build also precomputes a compact
per-shard **strict-AOI mask payload**: at write time every output shard gets an
`aoi_mask` boolean cell column — *"package, don't clip"*: edge shards keep
their full photon content, and the AOI selection becomes a read-time filter.

In [ ]:
from importlib.resources import files

from zagg.config import load_config
from zagg.grids import from_config

cfg_dir = files("zagg.configs")
cfg_tdigest = load_config(str(cfg_dir / "atl03_tdigest_serc.yaml"))
cfg_gain_bias = load_config(str(cfg_dir / "atl03_gain_bias_serc.yaml"))

grid = from_config(cfg_tdigest)
grid_gb = from_config(cfg_gain_bias)

# The reuse contract (#89): identical spatial signatures -> one shardmap serves both.
assert grid.spatial_signature() == grid_gb.spatial_signature()
grid.spatial_signature()

In [ ]:
from zagg.catalog import load_polygon
from zagg.catalog.shardmap import ShardMap

parts = load_polygon(AOI_GEOJSON)   # [(lats, lons)] ring parts of the SERC box

# region drives shard coverage; the strict-AOI mask defaults to the same
# polygon. backend="mortie" is the HEALPix-native intersection engine
# (backend="auto" upgrades to exact-S2 spherely when installed).
shardmap = ShardMap.build(catalog, grid, region=parts, backend="mortie")

print(f"{len(shardmap.shard_keys)} shards, "
      f"{shardmap.metadata['total_pairs']} granule-shard pairs, "
      f"built in {shardmap.metadata['build_wall_s']:.2f}s")
print(f"strict-AOI payload present: {shardmap.aoi_mask is not None}")

In [ ]:
import numpy as np

sm_path = DATA / "sm_serc_healpix_o10.json"
shardmap.to_json(str(sm_path))
print(f"{sm_path} ({sm_path.stat().st_size / 1024:.0f} KB)\n")

# Per-shard granule load + how much of each shard is inside the strict AOI.
# The stored payload is a compact sub-MOC; the grid expands it to a per-cell
# bool over the shard's children — the same expansion the worker does.
print(f"{'shard key':>20}  granules  in-AOI cells (of {len(grid.children(shardmap.shard_keys[0])):,})")
for key, grans, payload in zip(shardmap.shard_keys, shardmap.granules, shardmap.aoi_mask):
    mask = grid.aoi_mask_for_children(np.asarray(payload, dtype=np.uint64), grid.children(key))
    print(f"{key:>20}  {len(grans):8d}  {int(mask.sum()):12,}")

### Visualize the shardmap

Nine order-10 HEALPix shards cover the ~10 km SERC box. The static render below
colors each shard by its granule count over the strict-AOI polygon (red); the
following cell opens the same map interactively (`zagg.viz`, ipyleaflet) with
the granule footprints as a toggleable layer, exactly as in
[`shardmap_viewer.ipynb`](shardmap_viewer.ipynb).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize

fig, ax = plt.subplots(figsize=(7, 7))
counts = [len(g) for g in shardmap.granules]
norm = Normalize(vmin=min(counts), vmax=max(counts))
cmap = colormaps["viridis"]

for key, n in zip(shardmap.shard_keys, counts):
    poly = grid.shard_footprint(key)          # WGS84 (lon, lat), lon in 0..360
    lon, lat = np.asarray(poly.exterior.xy[0]), np.asarray(poly.exterior.xy[1])
    lon = np.where(lon > 180, lon - 360, lon)
    ax.fill(lon, lat, color=cmap(norm(n)), alpha=0.6, edgecolor="k", lw=0.6)
    ax.annotate(str(n), (lon.mean(), lat.mean()), ha="center", fontsize=9)

for lats, lons in parts:                       # the strict-AOI polygon
    ax.plot(np.asarray(lons), np.asarray(lats), "r-", lw=2, label="SERC AOI")

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
plt.colorbar(sm, ax=ax, label="granules per shard", shrink=0.8)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("SERC shardmap: 9 HEALPix o10 shards (label = granule count)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
from zagg.viz import show_shardmap

# Interactive viewer (mid-latitude AOI -> Web Mercator). Granule footprints and
# shard outlines are separate toggleable layers.
m = show_shardmap(str(sm_path), catalog=str(cat_path), zoom=11)
print("display CRS:", m.crs["name"])
m

## 3. The two run configs

Both configs ship with zagg (`zagg/configs/atl03_tdigest_serc.yaml`,
`zagg/configs/atl03_gain_bias_serc.yaml`). They share the read plan (planned
segment→photon IO, issue #43), the confidence filter, the output grid — and
therefore the shardmap — and differ only in the `aggregation` block:

| | `atl03_tdigest_serc` | `atl03_gain_bias_serc` |
|---|---|---|
| per-cell output | t-digest sketch of `h_ph` (ragged CSR, ~256 centroids) | 128-bin waveform counts (`uint32[128]`) |
| per-chunk output | — | `offset_h` + `gain_h` (the scale/offset frame) |
| location channel | **morton locations per centroid** (issue #87) | — |
| quantiles at read time | any (p5/median/p95/...) | binned only |

Three newer features to notice:

- **`output.aoi_mask: true`** (issue #101) — every output shard carries a
  per-cell in-AOI boolean; nothing is clipped at write time.
- **`data_source.index`** (issue #160) — the read path goes through a pluggable
  **virtual chunk-index**. The t-digest config uses the `inline` backend with
  `write_back: true`: each granule's HDF5 chunk map is built on the fly
  (~1 GET + ~20 ms) and **persisted** as a granule-keyed parquet manifest under
  the canonical sidecar prefix `zagg-index/<product>/<version>/`. That cache is
  grid- and AOI-independent — built once per granule, reusable by every future
  config.
- **`location: leaf_id`** on the t-digest field (issue #87) — each centroid also
  stores the deepest HEALPix cell (packed order-29 morton word) enclosing the
  photons merged into it, as a lossless `uint64` companion CSR array.

With `h5coro-hidefix` ≥ 0.2 installed, a later run consumes the manifests
directly by swapping the index block to the plugin-provided `sidecar` backend —
which is exactly how §5 runs the gain/bias aggregation:

```yaml
data_source:
  index:
    backend: sidecar                                    # from h5coro-hidefix
    store: s3://sliderule-public-cors/zagg-index/ATL03/007
    on_miss: fallback     # hierarchical read for granules not yet cached
```

(The gain/bias config as shipped carries `backend: inline` so it stays runnable
with zagg alone; the backend registry discovers `sidecar` through the
`zagg.index_backends` entry point whenever the plugin is present.)

In [ ]:
import yaml

print("tdigest  index block:", cfg_tdigest.data_source["index"])
print("gain_bias index block:", cfg_gain_bias.data_source["index"])
print()
print("located t-digest field:")
print(yaml.dump({"h_tdigest": cfg_tdigest.aggregation["variables"]["h_tdigest"]},
                default_flow_style=False))

## 4. Dispatch: the deployed path (copy-paste)

In production each shard is one AWS Lambda invocation: `agg(...)` fans the
shardmap out (one worker per shard), each worker reads its granules straight
from NSIDC's cloud holdings, and the workers write Zarr chunk regions + CSR
groups directly to the output bucket. The full run needs:

- **NASA Earthdata Login** (a `~/.netrc` entry for `urs.earthdata.nasa.gov`) —
  zagg exchanges it for NSIDC S3 credentials (`driver="s3"`, us-west-2) or an
  EDL bearer token (`driver="https"`, works anywhere);
- **AWS credentials** for the Lambda invoke + the output bucket write.

```python
from zagg.runner import agg

# 1) t-digest — also populates the granule-keyed chunk-manifest cache
#    (index: inline + write_back) under zagg-index/ATL03/007/.
summary_td = agg(
    cfg_tdigest,
    catalog="data/sm_serc_healpix_o10.json",
    backend="lambda",              # one worker per shard
    driver="s3",                   # direct NSIDC S3 reads (us-west-2)
    invocation="async",            # poll per-shard result objects (issue #151)
    overwrite=True,
)

# 2) gain/bias — SAME shardmap, different aggregation (issue #89).
summary_gb = agg(
    cfg_gain_bias,
    catalog="data/sm_serc_healpix_o10.json",
    backend="lambda",
    driver="s3",
    invocation="async",
    overwrite=True,
)
```

Both stores land under `s3://sliderule-public-cors/zagg-examples/serc/` (the
`output.store` values baked into the configs).

> **Note (July 2026):** the *deployed* Lambda still runs zagg 0.12.x, which
> predates the virtual-index (#160) and location-channel (#87) features these
> configs use — so the runs below execute **locally, in-process**
> (`backend="local"`) until the 0.14 layer is deployed. The local backend is
> the same worker code on a thread pool; outputs are identical.

## 5. Execute: two aggregations, one shardmap (run locally)

This is the run that produced the committed outputs and the published stores.
It reads from a **local granule cache** — the 59 catalog granules downloaded
once with `earthaccess` (~88 GB; the cache covers every granule in the
shardmap) — by pointing the shardmap's hrefs at the cache and swapping
h5coro's S3 driver for its `FileDriver`. No credentials are needed for any of
it. To reproduce, populate the cache first:

```python
import earthaccess

earthaccess.login()                # EDL prompt / ~/.netrc
results = earthaccess.search_data(
    short_name="ATL03", version="007",
    bounding_box=(-76.63, 38.84, -76.50, 38.94),   # SERC box
    temporal=("2018-10-13", "2025-06-01"),
)
earthaccess.download(results, GRANULE_CACHE)
```

In [ ]:
import json
import os

GRANULE_CACHE = Path(os.environ.get(
    "ZAGG_GRANULE_CACHE",
    Path.home() / "ignore" / "zagg_neon_atl03_test_shard" / "granules",
))
RUN_ROOT = Path(os.environ.get("ZAGG_RUN_ROOT", Path.home() / ".cache" / "zagg" / "serc_example"))
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Local-cache adapters: FileDriver instead of S3Driver, and no NSIDC
# credential exchange (local files need none).
import h5coro.s3driver as s3driver
from h5coro import filedriver

import zagg.runner as runner

s3driver.S3Driver = filedriver.FileDriver
runner.get_nsidc_s3_credentials = lambda: {}

# Point the shardmap's granule hrefs at the cache (the manifest is
# endpoint-neutral: id + s3 + https per granule).
sm_local = json.load(open(sm_path))
missing = 0
for shard_granules in sm_local["granules"]:
    for g in shard_granules:
        local = GRANULE_CACHE / g["id"]
        missing += not local.exists()
        g["s3"] = str(local)
sm_local_path = RUN_ROOT / "sm_serc_healpix_o10_local.json"
json.dump(sm_local, open(sm_local_path, "w"))
print(f"granule cache: {GRANULE_CACHE} — {missing} of "
      f"{sum(len(g) for g in sm_local['granules'])} hrefs missing")

In [ ]:
# Run 1: located t-digest. index: inline + write_back builds each granule's
# chunk manifest at read time and persists it — the store paths below mirror
# the canonical bucket layout, so publishing is a plain `aws s3 sync`.
cfg_tdigest.data_source["index"]["store"] = str(RUN_ROOT / "zagg-index" / "ATL03" / "007")

summary_td = runner.agg(
    cfg_tdigest,
    catalog=str(sm_local_path),
    store=str(RUN_ROOT / "tdigest.zarr"),
    backend="local",
    max_workers=4,
    overwrite=True,
)
{k: v for k, v in summary_td.items() if k != "results"}

### The virtual-index cache the run left behind

`write_back` persisted one parquet manifest per granule — the granule's full
HDF5 chunk map (byte offsets + sizes + decode metadata for every configured
dataset, PR #159's schema). The cache is keyed by granule id alone: it is
grid-, AOI- and config-independent, built once per granule, ever. Any tool
that can read parquet can address raw byte ranges in the archive without
touching HDF5 B-trees — and zagg's `sidecar` backend does exactly that,
next.

In [ ]:
import pandas as pd

index_dir = RUN_ROOT / "zagg-index" / "ATL03" / "007"
manifests = sorted(index_dir.glob("*.parquet"))
print(f"{len(manifests)} granule manifests under {index_dir.name}/")

mf = pd.read_parquet(manifests[0])
print(f"\n{manifests[0].name}: {len(mf)} chunks")
mf.head()

### Run 2: gain/bias, consuming the cache through `sidecar`

The second aggregation reuses the **same shardmap** (issue #89 — nothing is
re-intersected, no catalog is re-fetched) and reads through the **`sidecar`**
backend from `h5coro-hidefix` (≥ 0.2, discovered via the `zagg.index_backends`
entry point): each granule's manifest is fetched once, byte ranges are
addressed from it, and buffers are decoded by hidefix — HDF5 B-trees are never
walked. `on_miss: fallback` keeps the run robust if a granule's manifest were
missing. Backends are interchangeable by contract: `sidecar`, `inline`, and
`hierarchical` produce **byte-identical** stores — zagg's conformance gate
(`tests/test_index.py`), spot-verified on this data: an `inline` rerun of one
shard matches the sidecar output byte-for-byte in `waveform_counts` and
`gain_h`.

In [ ]:
from zagg.index import available_index_backends

print("index backends discovered:", sorted(available_index_backends()))

# Same config, cache-consuming index block (the shipped config says `inline`
# so it stays runnable with zagg alone — see section 3).
cfg_gain_bias.data_source["index"] = {
    "backend": "sidecar",
    "store": str(index_dir),          # published copy: s3://sliderule-public-cors/zagg-index/ATL03/007
    "on_miss": "fallback",
}

summary_gb = runner.agg(
    cfg_gain_bias,
    catalog=str(sm_local_path),
    store=str(RUN_ROOT / "gain_bias.zarr"),
    backend="local",
    max_workers=4,
    overwrite=True,
)
{k: v for k, v in summary_gb.items() if k != "results"}

In [ ]:
# Same shardmap + same confidence filter -> the two runs saw exactly the
# same photons, one through inline chunk maps, one through the sidecar cache.
assert summary_td["total_obs"] == summary_gb["total_obs"]
print(f"both aggregations processed {summary_td['total_obs']:,} photons "
      f"across {summary_td['cells_with_data']} shards")

### Publish

The stores and the manifest cache mirror the bucket layout, so publication is
two syncs (needs write access to `sliderule-public-cors`):

```bash
aws s3 sync "$ZAGG_RUN_ROOT/tdigest.zarr"   s3://sliderule-public-cors/zagg-examples/serc/tdigest.zarr
aws s3 sync "$ZAGG_RUN_ROOT/gain_bias.zarr" s3://sliderule-public-cors/zagg-examples/serc/gain_bias.zarr
aws s3 sync "$ZAGG_RUN_ROOT/zagg-index"     s3://sliderule-public-cors/zagg-index
```

Everything under `sliderule-public-cors` is anonymously readable — which is all
[`sponsor_read_only.ipynb`](sponsor_read_only.ipynb) needs.

## 6. Read-back

Everything below reads only the finished Zarr stores — it is the same code the
zero-credential companion notebook runs against the published bucket copies
(there the `store` argument is an anonymous S3 URL instead of a local path;
nothing else changes).

The store layout, under the `19` group (the `child_order` — one group per
resolution): dense per-cell arrays (`count`, `cell_ids`, `morton`, `aoi_mask`,
and for gain/bias `waveform_counts` + the per-chunk `offset_h`/`gain_h`), plus
one CSR subgroup per inner chunk for the ragged t-digest field
(`h_tdigest/<chunk>/{values,offsets,cell_ids,locations}`).

In [ ]:
import zarr

td_root = zarr.open_group(str(RUN_ROOT / "tdigest.zarr"), mode="r")["19"]
gb_root = zarr.open_group(str(RUN_ROOT / "gain_bias.zarr"), mode="r")["19"]

print("tdigest arrays:  ", sorted(td_root.array_keys()))
print("gain_bias arrays:", sorted(gb_root.array_keys()))
chunk_names = sorted(td_root["h_tdigest"].group_keys(), key=int)
print(f"t-digest CSR chunks: {len(chunk_names)} "
      f"({len(shardmap.shard_keys)} shards x 64 inner chunks)")

### Photon density and the strict-AOI mask

Per shard, the dense arrays are contiguous slices of the full-sphere cell
index space (`block_index * 4^(child-parent)`), so reading a shard is one
slice. The left panel shows every populated ~10 m cell — ICESat-2's ground
tracks criss-crossing the nine shards; the right panel applies the `aoi_mask`
column: *the same data*, selected down to the strict SERC polygon at read time
(issue #101's "package, don't clip").

In [ ]:
from mortie.tools import mort2geo

CELLS_PER_SHARD = 4 ** (grid.child_order - grid.parent_order)


def shard_slice(root, shard_key, *names):
    lo = grid.block_index(shard_key)[0] * CELLS_PER_SHARD
    return [root[n][lo:lo + CELLS_PER_SHARD] for n in names]


pts = {"lon": [], "lat": [], "count": [], "in_aoi": []}
for key in shardmap.shard_keys:
    count, aoi, mort = shard_slice(td_root, key, "count", "aoi_mask", "morton")
    nz = count > 0
    lat, lon = mort2geo(mort[nz])
    pts["lon"].append(np.where(lon > 180, lon - 360, lon))
    pts["lat"].append(lat)
    pts["count"].append(count[nz])
    pts["in_aoi"].append(aoi[nz])
pts = {k: np.concatenate(v) for k, v in pts.items()}

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
for ax, sel, title in [
    (axes[0], slice(None), f"all populated cells ({len(pts['count']):,})"),
    (axes[1], pts["in_aoi"], f"strict-AOI cells only ({int(pts['in_aoi'].sum()):,})"),
]:
    sc = ax.scatter(pts["lon"][sel], pts["lat"][sel], c=np.log10(pts["count"][sel]),
                    s=1.5, cmap="viridis")
    for lats, lons in parts:
        ax.plot(np.asarray(lons), np.asarray(lats), "r-", lw=1.5)
    ax.set_title(title)
    ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude")
plt.colorbar(sc, ax=axes, label="log10 photons per 10 m cell", shrink=0.8)
plt.show()

### T-digest products: any quantile, at read time

Each populated cell stores ~256 `(mean, weight)` centroids — enough to
evaluate *any* quantile after the fact. Over the SERC forest the p05→p95
spread of photon heights is a canopy-structure proxy: ground returns pin the
low quantiles, canopy top the high ones. No re-aggregation, no photon
re-reads — just the sketch.

In [ ]:
from zagg.csr import iter_csr_cells, read_csr
from zagg.stats.tdigest import quantile_from_tdigest

# The densest shard (48 granules) sits fully inside the AOI box.
DENSE_SHARD = max(zip(shardmap.shard_keys, shardmap.granules), key=lambda t: len(t[1]))[0]
h10 = grid.block_index(DENSE_SHARD)[0]
count, aoi, mort = shard_slice(td_root, DENSE_SHARD, "count", "aoi_mask", "morton")

med, spread, gidx = [], [], []
td_store_path = str(RUN_ROOT / "tdigest.zarr")
for h13 in range(h10 * 64, (h10 + 1) * 64):
    for cid, dig in iter_csr_cells(read_csr(td_store_path, f"19/h_tdigest/{h13}")):
        d = np.asarray(dig, np.float64)
        q05, q50, q95 = (quantile_from_tdigest(d, q) for q in (0.05, 0.50, 0.95))
        med.append(q50)
        spread.append(q95 - q05)
        gidx.append((h13 - h10 * 64) * 4096 + cid)
gidx = np.asarray(gidx)
keep = aoi[gidx]                       # strict-AOI cells only
lat, lon = mort2geo(mort[gidx][keep])
lon = np.where(lon > 180, lon - 360, lon)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, vals, label, cmap in [
    (axes[0], np.asarray(med)[keep], "median photon height (m)", "terrain"),
    (axes[1], np.asarray(spread)[keep], "p95 - p05 spread (m)", "magma"),
]:
    sc = ax.scatter(lon, lat, c=vals, s=4, cmap=cmap)
    plt.colorbar(sc, ax=ax, label=label, shrink=0.85)
    ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude")
fig.suptitle(f"shard {DENSE_SHARD}: quantiles evaluated from the stored digests "
             f"({int(keep.sum()):,} in-AOI cells)")
plt.tight_layout()
plt.show()

### Cross-check: t-digest reconstruction vs the binned waveform

The two aggregations saw identical photons, so a cell's t-digest — rasterized
over *the gain/bias run's own window* (`offset_h - 27*gain_h`, 128 bins of
`gain_h` metres) — should reproduce that cell's stored `waveform_counts`. We
pick the most photon-heavy in-AOI cell among the shard's finest-gain chunks
and overlay the two. (Chunk gains here are several metres, not the 0.5 m
floor: the confidence filter keeps flag-0 background photons, whose
atmospheric spread sets each chunk's `max - min` range — see the config
header notes.)

In [ ]:
from zagg.readers import rasterize_cell

gain_arr = gb_root["gain_h"][h10 * 64:(h10 + 1) * 64]
off_arr = gb_root["offset_h"][h10 * 64:(h10 + 1) * 64]

# densest in-AOI cell within the shard's finest-gain (sharpest-window) chunks;
# in-AOI preferred, any cell as fallback.
populated = [k for k in range(64) if gain_arr[k] > 0]
g_min = min(gain_arr[k] for k in populated)
best = fallback = None
for k in populated:
    if gain_arr[k] > g_min:
        continue
    h13 = h10 * 64 + k
    for cid, dig in iter_csr_cells(read_csr(td_store_path, f"19/h_tdigest/{h13}")):
        g = k * 4096 + cid
        w = np.asarray(dig)[:, 1].sum()
        cand = (w, h13, k, cid, np.asarray(dig, np.float64))
        if fallback is None or w > fallback[0]:
            fallback = cand
        if aoi[g] and (best is None or w > best[0]):
            best = cand
n_ph, h13, k, cid, dig = best or fallback
gain, off = float(gain_arr[k]), float(off_arr[k])

waveform = gb_root["waveform_counts"][h13 * 4096 + cid]
z_lo = off - 27.0 * gain
recon = rasterize_cell(dig, z_lo, gain, 128)
z = z_lo + gain * (np.arange(128) + 0.5)
r = np.corrcoef(waveform, recon)[0, 1]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(z, waveform, where="mid", lw=1.8, label="gain_bias: stored 128-bin waveform")
ax.step(z, recon, where="mid", lw=1.8, alpha=0.75,
        label="t-digest: reconstructed from ~256 centroids")
ax.axvline(off, color="gray", ls=":", lw=1, label="offset_h (DEM anchor, bin 28)")
ax.set_xlabel("elevation (m)")
ax.set_ylabel("photons per bin")
ax.set_title(f"chunk {h13}, cell {cid}: {int(n_ph)} photons, "
             f"{gain} m bins — r = {r:.5f}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"photon totals — waveform: {int(waveform.sum())}, "
      f"digest: {recon.sum():.1f}, shard reads agree: {int(n_ph)}")

### The morton location channel: sub-cell structure

The located t-digest (issue #87) stores one packed order-29 morton word per
centroid — the deepest HEALPix cell enclosing the photons merged into that
centroid (~2 cm at order 29). Decoding the words for the cell above shows
*where inside the ~10 m cell* each centroid's photons sit: the along-track
stripe of the beam, with height structure resolved along it. The channel is
lossless in `uint64` (an order-29 word exceeds float64's exact-integer range)
and composes directly with mortie's query primitives
(`clip2order`, `common_ancestor`, `mort2geo`).

In [ ]:
from zagg.readers import read_locations

locs = next(
    l for m, c, l in read_locations(str(RUN_ROOT / "tdigest.zarr"), "19/h_tdigest")
    if m == h13 and c == cid
)
lat_c, lon_c = mort2geo(locs)
lon_c = np.where(lon_c > 180, lon_c - 360, lon_c)
dx = (lon_c - lon_c.mean()) * 111_320 * np.cos(np.deg2rad(lat_c.mean()))
dy = (lat_c - lat_c.mean()) * 111_320

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sc = ax.scatter(dx, dy, c=dig[:, 0], s=22, cmap="terrain")
plt.colorbar(sc, label="centroid mean height (m)")
ax.set_xlabel("east offset (m)")
ax.set_ylabel("north offset (m)")
ax.set_title(f"cell {cid}: {len(locs)} centroid locations inside one ~10 m cell")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()
print(f"location span: {np.ptp(dx):.1f} m east-west x {np.ptp(dy):.1f} m north-south")

## 7. Where to go from here

- **Read the published products with no credentials** —
  [`sponsor_read_only.ipynb`](sponsor_read_only.ipynb) repeats section 6
  against `s3://sliderule-public-cors/zagg-examples/serc/` anonymously (and
  fetches a granule manifest from `zagg-index/ATL03/007/`), end-to-end on
  Binder.
- **Scale it up** — swap `backend="local"` for `backend="lambda"` (section 4)
  and the same shardmap fans out one worker per shard; the committed benchmark
  matrix (`tests/data/benchmark/`) runs these exact config templates at orders
  8–11.
- **Reuse the caches** — the STAC-geoparquet catalog, the shardmap, and the
  granule-keyed chunk manifests are all build-once artifacts: new aggregations
  start at the `agg()` call.